<a href="https://colab.research.google.com/github/tbruffy/financial-economics-ps5/blob/main/ps5_q1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Economics I — Problem Set 5

## Problem 1(d)

Simulation of the consumption-based asset-pricing model over
100,000 annual observations.

In [2]:
import numpy as np
import pandas as pd

# Simulation settings
T = 100_000
SEED = 42

# Annual calibration from Problem 1(c)
sigma = 0.036
mu = 0.018
gamma = 10.0
beta = 0.99

print(f"Annual observations: {T:,}")
print(f"Random seed: {SEED}")
print(f"sigma={sigma}, mu={mu}, gamma={gamma}, beta={beta}")


Annual observations: 100,000
Random seed: 42
sigma=0.036, mu=0.018, gamma=10.0, beta=0.99


In [3]:
# Generate one reproducible sequence of annual shocks
rng = np.random.default_rng(SEED)
epsilon = rng.standard_normal(T)

# Inspect the draws
print("First five shocks:", epsilon[:5])
print(f"Number of shocks: {epsilon.size:,}")
print(f"Sample mean: {np.mean(epsilon):.6f}")
print(f"Sample standard deviation: {np.std(epsilon, ddof=1):.6f}")

First five shocks: [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519]
Number of shocks: 100,000
Sample mean: -0.004233
Sample standard deviation: 1.003708


In [4]:
# Log consumption growth
delta_c = mu + sigma * epsilon

# Log stochastic discount factor
m = np.log(beta) - gamma * delta_c

# Constant log riskfree rate
r_f_value = (
    -np.log(beta)
    + gamma * mu
    - 0.5 * gamma**2 * sigma**2
)
r_f = np.full(T, r_f_value)

# Log equity return
r_d = (
    -np.log(beta)
    + gamma * mu
    - 0.5 * (1 - gamma)**2 * sigma**2
    + sigma * epsilon
)

# Organize the four series into a table
data = pd.DataFrame({
    "r_f": r_f,
    "delta_c": delta_c,
    "m": m,
    "r_d": r_d,
})

data.head()

,r_f,delta_c,m,r_d
0,0.12525,0.028970,-0.299748,0.148532
1,0.12525,-0.019439,0.184344,0.100123
2,0.12525,0.045016,-0.460213,0.164579
3,0.12525,0.051860,-0.528654,0.171423
4,0.12525,-0.052237,0.512322,0.067325


In [5]:
# Sample standard deviation
def sample_std(x):
    x = np.asarray(x, dtype=float)

    if np.all(x == x[0]):
        return 0.0

    return float(np.std(x, ddof=1))


# First sample autocorrelation
def acf1(x):
    x = np.asarray(x, dtype=float)

    if np.all(x == x[0]):
        return np.nan

    z = x - np.mean(x)

    return float(
        np.dot(z[1:], z[:-1]) / np.dot(z, z)
    )


# Apply both functions to each column
statistics = pd.DataFrame({
    "Standard deviation": data.apply(sample_std),
    "First autocorrelation": data.apply(acf1),
})

print(
    statistics.to_string(
        float_format=lambda x: f"{x:.8f}",
        na_rep="N/A",
    )
)

         Standard deviation  First autocorrelation
r_f              0.00000000                    N/A
delta_c          0.03613350             0.00129805
m                0.36133500             0.00129805
r_d              0.03613350             0.00129805


In [6]:
correlation = np.corrcoef(delta_c, r_d)[0, 1]

print(
    "Consumption-growth/equity-return correlation: "
    f"{correlation:.8f}"
)

Consumption-growth/equity-return correlation: 1.00000000


Consumption growth and equity returns are perfectly positively correlated because they depend on the same innovation and differ only by a constant. This follows from the model’s constant price–dividend ratio.